In [1]:
pip install nba_api

Note: you may need to restart the kernel to use updated packages.


In [61]:
from nba_api.stats.endpoints import teamgamelogs
from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguedashteamstats
import pandas as pd

In [9]:
# Find team IDs

nba_teams = teams.get_teams()

okc = [t for t in nba_teams if t['abbreviation'] == 'OKC'][0]
sas = [t for t in nba_teams if t['abbreviation'] == 'SAS'][0]

okc_id = okc['id']
sas_id = sas['id']
print(okc)
print(sas)
print(okc_id)
print(sas_id)

{'id': 1610612760, 'full_name': 'Oklahoma City Thunder', 'abbreviation': 'OKC', 'nickname': 'Thunder', 'city': 'Oklahoma City', 'state': 'Oklahoma', 'year_founded': 1967}
{'id': 1610612759, 'full_name': 'San Antonio Spurs', 'abbreviation': 'SAS', 'nickname': 'Spurs', 'city': 'San Antonio', 'state': 'Texas', 'year_founded': 1976}
1610612760
1610612759


In [39]:
okc_season_gamelog = teamgamelogs.TeamGameLogs(team_id_nullable = okc_id, season_nullable= '2025-26', season_type_nullable='Regular Season')
okc_playoff_gamelog = teamgamelogs.TeamGameLogs(team_id_nullable = okc_id, season_nullable = '2025-26', season_type_nullable='Playoffs')

okc_playoff = okc_playoff_gamelog.get_data_frames()[0]
okc_season = okc_season_gamelog.get_data_frames()[0]


sas_season_gamelog = teamgamelogs.TeamGameLogs(team_id_nullable = sas_id, season_nullable= '2025-26', season_type_nullable='Regular Season')
sas_playoff_gamelog = teamgamelogs.TeamGameLogs(team_id_nullable = sas_id, season_nullable = '2025-26', season_type_nullable='Playoffs')

sas_playoff = sas_playoff_gamelog.get_data_frames()[0]
sas_season = sas_season_gamelog.get_data_frames()[0]

#print(okc_season.head())
#print(okc_playoff.head())

print(sas_season.head())
print(sas_playoff.head())

  SEASON_YEAR     TEAM_ID TEAM_ABBREVIATION          TEAM_NAME     GAME_ID  \
0     2025-26  1610612759               SAS  San Antonio Spurs  0022501197   
1     2025-26  1610612759               SAS  San Antonio Spurs  0022501180   
2     2025-26  1610612759               SAS  San Antonio Spurs  0022501161   
3     2025-26  1610612759               SAS  San Antonio Spurs  0022501146   
4     2025-26  1610612759               SAS  San Antonio Spurs  0022501130   

             GAME_DATE      MATCHUP WL   MIN  FGM  ...  AST_RANK  TOV_RANK  \
0  2026-04-12T00:00:00  SAS vs. DEN  L  48.0   45  ...        48         6   
1  2026-04-10T00:00:00  SAS vs. DAL  W  48.0   50  ...        26        18   
2  2026-04-08T00:00:00  SAS vs. POR  W  48.0   43  ...        41        66   
3  2026-04-06T00:00:00  SAS vs. PHI  W  48.0   44  ...        22        18   
4  2026-04-04T00:00:00    SAS @ DEN  L  53.0   47  ...        35        18   

   STL_RANK  BLK_RANK  BLKA_RANK  PF_RANK  PFD_RANK  PTS_RANK 

In [82]:
def clean_team_data(team_data,  is_playoff):
    cleaned_team_data = team_data[['GAME_DATE', 'MATCHUP', 'WL','PTS','PLUS_MINUS', 'FGA', 'FG_PCT']].copy()
    cleaned_team_data['GAME_DATE'] = pd.to_datetime(cleaned_team_data['GAME_DATE']).dt.date
    cleaned_team_data['POINTS_AGAINST'] = team_data['PTS'] - team_data['PLUS_MINUS']
    cleaned_team_data['PACE'] = team_data['FGA'] + (0.44 * team_data['FTA'])-team_data['OREB'] +team_data['TOV']
    cleaned_team_data['IS_HOME'] = team_data['MATCHUP'].apply(lambda x: 1 if '@' not in x else 0)
    cleaned_team_data['PLAYOFF'] = True if is_playoff else False
    return cleaned_team_data

In [84]:
# Collect Season and Playoff per game data

print(type(okc_season['MATCHUP']))
okc_cleaned_season = clean_team_data(okc_season,is_playoff=False)
okc_cleaned_playoff = clean_team_data(okc_playoff,is_playoff=True)
sas_cleaned_season = clean_team_data(sas_season,is_playoff=False)
sas_cleaned_playoff = clean_team_data(sas_playoff,is_playoff=True)

okc_data = pd.concat([okc_cleaned_season, okc_cleaned_playoff], ignore_index= True).sort_values(by= 'GAME_DATE', ascending = True, ignore_index = True)
sas_data = pd.concat([sas_cleaned_season, sas_cleaned_playoff], ignore_index= True).sort_values(by= 'GAME_DATE', ascending = True, ignore_index = True)

print(sas_data[-15:])

<class 'pandas.core.series.Series'>
     GAME_DATE      MATCHUP WL  PTS  PLUS_MINUS  FGA  FG_PCT  POINTS_AGAINST  \
79  2026-04-08  SAS vs. POR  W  112        11.0   88   0.489           101.0   
80  2026-04-10  SAS vs. DAL  W  139        19.0   92   0.543           120.0   
81  2026-04-12  SAS vs. DEN  L  118       -10.0   99   0.455           128.0   
82  2026-04-19  SAS vs. POR  W  111        13.0   84   0.476            98.0   
83  2026-04-21  SAS vs. POR  L  103        -3.0   86   0.442           106.0   
84  2026-04-24    SAS @ POR  W  120        12.0   87   0.471           108.0   
85  2026-04-26    SAS @ POR  W  114        21.0   87   0.494            93.0   
86  2026-04-28  SAS vs. POR  W  114        19.0   75   0.547            95.0   
87  2026-05-04  SAS vs. MIN  L  102        -2.0   87   0.448           104.0   
88  2026-05-06  SAS vs. MIN  W  133        38.0   90   0.500            95.0   
89  2026-05-08    SAS @ MIN  W  115         7.0   85   0.459           108.0   
90  

In [86]:
#Save dataframes into csv
okc_data.to_csv('../data/raw/okc_2526')
sas_data.to_csv('../data/raw/sas_2526')